# GPU LSTM training — Las Vegas and Phoenix energy deviations

This notebook is the training stage for `LSTM_LasVegas_Phoenix_Data_Preprocessing_BASELINE_DEVIATION.ipynb`. Run that preprocessing notebook first. This notebook reads its saved split files and **does not modify preprocessing**.

It trains Models 1–4 sequentially for each city. The target is `energy_deviation`; final energy is reconstructed as `energy_baseline + predicted_energy_deviation`.

In [1]:
from __future__ import annotations
import gc, json, os, random, time
from datetime import datetime
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import regularizers

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)

BASE_DIR = Path.cwd().parent
PREPROCESSED_DIR = BASE_DIR / "processed_lstm_baseline_deviation"
RESULTS_DIR = BASE_DIR / "LSTM_GPU_TRAINING_RESULTS"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SEQUENCE_LENGTH = 24
BATCH_SIZE = 64
EPOCHS = 100
PATIENCE = 10
LEARNING_RATE = 1e-3

CITY_KEYS = {"Las Vegas": "las_vegas", "Phoenix": "phoenix"}
MODELS = {
    "Model_1": ["temperature_c", "dew_point_c", "station_pressure_kpa", "wind_speed_ms", "wind_direction_deg"],
    "Model_2": ["temperature_c", "dew_point_c", "station_pressure_kpa", "wind_speed_ms", "wind_direction_deg", "precipitable_water_mm"],
    "Model_3": ["temperature_c", "dew_point_c", "station_pressure_kpa", "wind_speed_ms", "wind_direction_deg", "precipitable_water_mm", "broadband_aerosol_optical_depth"],
    "Model_4": ["temperature_c", "dew_point_c", "station_pressure_kpa", "wind_speed_ms", "wind_direction_deg", "precipitable_water_mm", "broadband_aerosol_optical_depth", "cooling_degree_hour"],
}
EXPECTED_COUNTS = {"Model_1": 5, "Model_2": 6, "Model_3": 7, "Model_4": 8}
print("Preprocessing outputs:", PREPROCESSED_DIR.resolve())
print("Training results:", RESULTS_DIR.resolve())

Preprocessing outputs: D:\Downloads\BTP\model_training\lstm_training_3\processed_lstm_baseline_deviation
Training results: D:\Downloads\BTP\model_training\lstm_training_3\LSTM_GPU_TRAINING_RESULTS


## 1. GPU confirmation

This controlled overnight experiment stops if TensorFlow cannot access a GPU, rather than silently using the CPU.

In [2]:
print("TensorFlow version:", tf.__version__)
physical_gpus = tf.config.list_physical_devices("GPU")
print("Physical GPUs:", physical_gpus)
if not physical_gpus:
    raise RuntimeError("No TensorFlow GPU detected. Configure TensorFlow/CUDA before starting this overnight GPU run.")
for gpu in physical_gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as exc:
        print("GPU memory-growth configuration:", exc)
print("Logical GPUs:", tf.config.list_logical_devices("GPU"))
with tf.device("/GPU:0"):
    probe = tf.reduce_sum(tf.ones((1024, 1024), dtype=tf.float32))
print("GPU execution probe:", float(probe.numpy()))

TensorFlow version: 2.10.0
Physical GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Logical GPUs: [LogicalDevice(name='/device:GPU:0', device_type='GPU')]
GPU execution probe: 1048576.0


## 2. Load saved preprocessing outputs and perform final checks

The saved baseline/deviation split CSVs are the source of truth. No dataframe is passed between notebook kernels. Leap-day omissions are expected only as the preprocessing pipeline’s February-28-to-March-1 transition; generated sequences must never cross them.

In [3]:
REQUIRED_BASE = {"timestamp", "hourly_load_mw", "energy_baseline", "energy_deviation"}
ALL_FEATURES = sorted(set(feature for fs in MODELS.values() for feature in fs))

def load_saved_city(city, key):
    city_dir = PREPROCESSED_DIR / key
    if not city_dir.exists():
        raise FileNotFoundError(f"{city}: preprocessing output directory missing: {city_dir.resolve()}")
    parts = {}
    for split, file_split in [("train", "train"), ("validation", "val"), ("test", "test")]:
        path = city_dir / f"{key}_{file_split}_baseline_deviation.csv"
        if not path.exists():
            raise FileNotFoundError(f"{city}: missing saved preprocessing output {path}")
        d = pd.read_csv(path, parse_dates=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
        missing = REQUIRED_BASE.union(ALL_FEATURES).difference(d.columns)
        if missing:
            raise ValueError(f"{city} {split}: required preprocessing columns missing: {sorted(missing)}")
        if d.timestamp.isna().any() or d.timestamp.duplicated().any():
            raise ValueError(f"{city} {split}: invalid or duplicate timestamps")
        numeric = ["hourly_load_mw", "energy_baseline", "energy_deviation", *ALL_FEATURES]
        if not np.isfinite(d[numeric].to_numpy(dtype=np.float64)).all():
            raise ValueError(f"{city} {split}: NaN/Inf found in target, baseline, or model inputs")
        reconstruction_error = np.abs(d.hourly_load_mw.to_numpy() - (d.energy_baseline.to_numpy() + d.energy_deviation.to_numpy())).max()
        if reconstruction_error > 1e-8:
            raise ValueError(f"{city} {split}: hourly_load_mw != energy_baseline + energy_deviation; max error={reconstruction_error}")
        parts[split] = d
    if not (parts["train"].timestamp.max() < parts["validation"].timestamp.min() < parts["test"].timestamp.min()):
        raise ValueError(f"{city}: chronological train/validation/test separation failed")
    return parts

def expected_gap_only(ts):
    diffs = ts.sort_values().diff().dropna()
    previous = ts.sort_values().iloc[:-1].reset_index(drop=True)
    current = ts.sort_values().iloc[1:].reset_index(drop=True)
    expected_leap_gap = ((previous.dt.month == 2) & (previous.dt.day == 28) & (previous.dt.hour == 23) & (current.dt.month == 3) & (current.dt.day == 1) & (current.dt.hour == 0))
    unexpected = (diffs.reset_index(drop=True) != pd.Timedelta(hours=1)) & ~expected_leap_gap
    return int(unexpected.sum()), int(expected_leap_gap.sum())

saved = {city: load_saved_city(city, key) for city, key in CITY_KEYS.items()}
summary = []
for city, parts in saved.items():
    combined = pd.concat(parts.values(), ignore_index=True).sort_values("timestamp")
    unexpected, leap = expected_gap_only(combined.timestamp)
    if unexpected:
        raise ValueError(f"{city}: {unexpected} unexpected timestamp gaps in preprocessing output")
    for name, fs in MODELS.items():
        if len(fs) != EXPECTED_COUNTS[name] or len(set(fs)) != len(fs):
            raise ValueError(f"{name}: incorrect feature configuration")
    summary.append({"city": city, "train_rows": len(parts["train"]), "validation_rows": len(parts["validation"]), "test_rows": len(parts["test"]), "expected_leap_gaps": leap, "target": "energy_deviation", "reconstruction": "PASS"})
display(pd.DataFrame(summary))
print("FINAL PRE-TRAINING VALIDATION PASSED: saved splits, target, baseline reconstruction, features, chronology, and timestamps are valid.")

,city,train_rows,validation_rows,test_rows,expected_leap_gaps,target,reconstruction
0,Las Vegas,49056,10512,10512,8,energy_deviation,PASS
1,Phoenix,49056,10512,10512,8,energy_deviation,PASS


FINAL PRE-TRAINING VALIDATION PASSED: saved splits, target, baseline reconstruction, features, chronology, and timestamps are valid.


## 3. Gap-safe memory-efficient datasets and fixed model architecture

Each feature scaler and deviation scaler is fit on that city’s training split only. The `tf.data` generator emits only a batch at a time, avoiding a full 3-D sequence array. The exact architecture remains fixed across all four models.

In [4]:
def fit_and_transform(parts, features):
    train = parts["train"]
    x_scaler, y_scaler = StandardScaler(), StandardScaler()
    x_scaler.fit(train[features].to_numpy(dtype=np.float32))
    y_scaler.fit(train[["energy_deviation"]].to_numpy(dtype=np.float32))
    transformed = {}
    for split, d in parts.items():
        transformed[split] = {
            "x": x_scaler.transform(d[features].to_numpy(dtype=np.float32)).astype(np.float32),
            "y": y_scaler.transform(d[["energy_deviation"]].to_numpy(dtype=np.float32)).ravel().astype(np.float32),
            "frame": d.reset_index(drop=True),
        }
    return transformed, x_scaler, y_scaler

def sequence_end_indices(frame):
    ts = frame.timestamp.astype("int64").to_numpy()
    possible = np.arange(SEQUENCE_LENGTH - 1, len(frame))
    span = (SEQUENCE_LENGTH - 1) * 3_600_000_000_000
    valid = possible[ts[possible] - ts[possible - SEQUENCE_LENGTH + 1] == span]
    if not len(valid):
        raise ValueError("No valid 24-hour sequences; inspect timestamp continuity")
    return valid

def make_dataset(x, y, ends, shuffle):
    order = np.array(ends, copy=True)
    def generate():
        if shuffle:
            np.random.shuffle(order)
        for start in range(0, len(order), BATCH_SIZE):
            batch_ends = order[start:start + BATCH_SIZE]
            xb = np.stack([x[i - SEQUENCE_LENGTH + 1:i + 1] for i in batch_ends]).astype(np.float32, copy=False)
            yield xb, y[batch_ends].astype(np.float32, copy=False)
    signature = (tf.TensorSpec(shape=(None, SEQUENCE_LENGTH, x.shape[1]), dtype=tf.float32), tf.TensorSpec(shape=(None,), dtype=tf.float32))
    return tf.data.Dataset.from_generator(generate, output_signature=signature).prefetch(tf.data.AUTOTUNE)

def build_lstm(feature_count):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(SEQUENCE_LENGTH, feature_count)),
        tf.keras.layers.LSTM(64, return_sequences=True, dropout=0.3, kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.LSTM(32, dropout=0.3, kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="linear"),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss="mse", metrics=["mae"])
    return model

for city, parts in saved.items():
    for model_name, features in MODELS.items():
        scaled, _, _ = fit_and_transform(parts, features)
        ends = {split: sequence_end_indices(item["frame"]) for split, item in scaled.items()}
        xb, yb = next(iter(make_dataset(scaled["train"]["x"], scaled["train"]["y"], ends["train"][:BATCH_SIZE], False)))
        assert xb.shape[1:] == (SEQUENCE_LENGTH, len(features)) and len(features) == EXPECTED_COUNTS[model_name]
print("Passed: train-only scaling, sequence length 24, exact feature counts, and gap-safe tf.data batches.")

Passed: train-only scaling, sequence length 24, exact feature counts, and gap-safe tf.data batches.


## 4. Sequential training, evaluation, plots, and durable outputs

Runs Model 1 through Model 4 in order. Each city/model is completed and saved before the next starts. Test metrics are the primary comparison metric.

In [5]:
def metric_dict(actual, predicted):
    return {"mae": float(mean_absolute_error(actual, predicted)), "rmse": float(mean_squared_error(actual, predicted) ** 0.5), "r2": float(r2_score(actual, predicted))}

def save_plots(out, history, actual_dev, pred_dev, actual_energy, pred_energy):
    plt.figure(figsize=(7,4)); plt.plot(history.history["loss"], label="train"); plt.plot(history.history["val_loss"], label="validation"); plt.xlabel("Epoch"); plt.ylabel("MSE loss"); plt.title("Training history"); plt.legend(); plt.tight_layout(); plt.savefig(out/"training_validation_loss.png", dpi=140); plt.close()
    for actual, predicted, title, filename in [(actual_energy,pred_energy,"Actual vs predicted final energy","actual_vs_predicted_final_energy.png"),(actual_dev,pred_dev,"Actual vs predicted deviation","actual_vs_predicted_deviation.png")]:
        plt.figure(figsize=(5,5)); plt.scatter(actual, predicted, s=5, alpha=.35); lo=min(actual.min(),predicted.min()); hi=max(actual.max(),predicted.max()); plt.plot([lo,hi],[lo,hi],"r--"); plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.title(title); plt.tight_layout(); plt.savefig(out/filename,dpi=140); plt.close()
    plt.figure(figsize=(7,4)); plt.hist(actual_energy-pred_energy, bins=60); plt.xlabel("Final energy error (MW)"); plt.title("Final-energy prediction error distribution"); plt.tight_layout(); plt.savefig(out/"final_energy_error_distribution.png",dpi=140); plt.close()

all_metrics=[]; all_test_predictions=[]
for model_name, features in MODELS.items():
    print(f"\n{'='*75}\nStarting {model_name}: {len(features)} features\n{'='*75}", flush=True)
    for city, parts in saved.items():
        started=time.time(); out=RESULTS_DIR/model_name/city.replace(" ","_"); out.mkdir(parents=True,exist_ok=True)
        print(f"{city}: training started", flush=True)
        tf.keras.backend.clear_session(); gc.collect(); tf.keras.utils.set_random_seed(SEED)
        scaled, x_scaler, y_scaler = fit_and_transform(parts, features)
        ends={split:sequence_end_indices(item["frame"]) for split,item in scaled.items()}
        datasets={split:make_dataset(item["x"],item["y"],ends[split],split=="train") for split,item in scaled.items()}
        with tf.device("/GPU:0"):
            model=build_lstm(len(features))
            history=model.fit(datasets["train"],validation_data=datasets["validation"],epochs=EPOCHS,verbose=2,callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=PATIENCE,restore_best_weights=True,verbose=1),tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=.5,patience=5,min_lr=1e-6,verbose=1),tf.keras.callbacks.CSVLogger(out/"training_history.csv"),tf.keras.callbacks.ModelCheckpoint(out/"best_model.keras",monitor="val_loss",save_best_only=True,verbose=1),tf.keras.callbacks.TerminateOnNaN()])
        model.save(out/"final_model.keras"); joblib.dump({"feature_scaler":x_scaler,"deviation_target_scaler":y_scaler},out/"scalers.joblib")
        metadata={"city":city,"model":model_name,"features":features,"feature_count":len(features),"target":"energy_deviation","sequence_length":SEQUENCE_LENGTH,"batch_size":BATCH_SIZE,"baseline_source":"saved city-specific energy_baseline from preprocessing","epochs_run":len(history.history["loss"]),"best_validation_loss":float(min(history.history["val_loss"]))}
        (out/"metadata.json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")
        split_metrics=[]
        for split in ["validation","test"]:
            pred_scaled=model.predict(datasets[split],verbose=0).ravel(); frame=scaled[split]["frame"].iloc[ends[split]].reset_index(drop=True)
            actual_dev=frame.energy_deviation.to_numpy(); pred_dev=y_scaler.inverse_transform(pred_scaled.reshape(-1,1)).ravel(); baseline=frame.energy_baseline.to_numpy(); actual_energy=frame.hourly_load_mw.to_numpy(); pred_energy=baseline+pred_dev
            predictions=pd.DataFrame({"timestamp":frame.timestamp,"actual_energy":actual_energy,"energy_baseline":baseline,"actual_energy_deviation":actual_dev,"predicted_energy_deviation":pred_dev,"predicted_final_energy":pred_energy,"deviation_error":actual_dev-pred_dev,"final_energy_error":actual_energy-pred_energy})
            predictions.to_csv(out/f"{split}_predictions.csv",index=False)
            result={"city":city,"model":model_name,"split":split,"number_of_features":len(features),"feature_list":" | ".join(features),"training_epochs":len(history.history["loss"]),"best_validation_loss":float(min(history.history["val_loss"])),**{"deviation_"+k:v for k,v in metric_dict(actual_dev,pred_dev).items()},**{"final_energy_"+k:v for k,v in metric_dict(actual_energy,pred_energy).items()}}
            split_metrics.append(result); all_metrics.append(result)
            if split=="test": all_test_predictions.append(predictions.assign(city=city,model=model_name)); save_plots(out,history,actual_dev,pred_dev,actual_energy,pred_energy)
        pd.DataFrame(split_metrics).to_csv(out/"metrics.csv",index=False)
        print(f"{city}: validation complete; test complete; results saved ({(time.time()-started)/60:.1f} min)",flush=True)

metrics=pd.DataFrame(all_metrics); metrics.to_csv(RESULTS_DIR/"all_metrics_by_split.csv",index=False)
test_predictions=pd.concat(all_test_predictions,ignore_index=True); test_predictions.to_csv(RESULTS_DIR/"all_test_predictions.csv",index=False)
comparison=metrics.query("split == 'test'").sort_values(["city","final_energy_rmse"]).reset_index(drop=True); comparison.to_csv(RESULTS_DIR/"final_test_model_comparison.csv",index=False)
display(comparison[["city","model","number_of_features","feature_list","deviation_mae","deviation_rmse","deviation_r2","final_energy_mae","final_energy_rmse","final_energy_r2","training_epochs","best_validation_loss"]])
best_by_city=comparison.loc[comparison.groupby("city").final_energy_rmse.idxmin()].sort_values("city"); overall=comparison.sort_values("final_energy_rmse").iloc[0]
print("\nBEST MODEL FOR EACH CITY (lowest held-out test final-energy RMSE):"); display(best_by_city[["city","model","final_energy_rmse","final_energy_r2"]])
print(f"Overall best feature configuration: {overall.model} ({overall.number_of_features} features), {overall.city}; test final-energy RMSE={overall.final_energy_rmse:.3f}")
for model_name in MODELS:
    vals=comparison[comparison.model.eq(model_name)].set_index("city").final_energy_r2.to_dict(); print(f"{model_name}: Las Vegas R² = {vals.get('Las Vegas',float('nan')):.4f}; Phoenix R² = {vals.get('Phoenix',float('nan')):.4f}")
print("\nComplete. Results:",RESULTS_DIR.resolve())


Starting Model_1: 5 features
Las Vegas: training started
Epoch 1/100

Epoch 1: val_loss improved from inf to 1.73197, saving model to d:\Downloads\BTP\model_training\lstm_training_3\LSTM_GPU_TRAINING_RESULTS\Model_1\Las_Vegas\best_model.keras
766/766 - 32s - loss: 0.9234 - mae: 0.6453 - val_loss: 1.7320 - val_mae: 0.8899 - lr: 0.0010 - 32s/epoch - 41ms/step
Epoch 2/100

Epoch 2: val_loss did not improve from 1.73197
766/766 - 16s - loss: 0.8713 - mae: 0.6311 - val_loss: 1.7676 - val_mae: 0.9057 - lr: 0.0010 - 16s/epoch - 20ms/step
Epoch 3/100

Epoch 3: val_loss did not improve from 1.73197
766/766 - 15s - loss: 0.8486 - mae: 0.6247 - val_loss: 1.7601 - val_mae: 0.9016 - lr: 0.0010 - 15s/epoch - 20ms/step
Epoch 4/100

Epoch 4: val_loss did not improve from 1.73197
766/766 - 15s - loss: 0.8347 - mae: 0.6195 - val_loss: 1.7829 - val_mae: 0.9076 - lr: 0.0010 - 15s/epoch - 20ms/step
Epoch 5/100

Epoch 5: val_loss did not improve from 1.73197
766/766 - 15s - loss: 0.8207 - mae: 0.6136 - val

,city,model,number_of_features,feature_list,deviation_mae,deviation_rmse,deviation_r2,final_energy_mae,final_energy_rmse,final_energy_r2,training_epochs,best_validation_loss
0,Las Vegas,Model_2,6,temperature_c | dew_point_c | station_pressure...,314.363733,423.045671,-0.059587,314.363733,423.045671,0.670028,11,1.688800
1,Las Vegas,Model_1,5,temperature_c | dew_point_c | station_pressure...,323.388930,425.712968,-0.072991,323.388930,425.712968,0.665854,11,1.731967
2,Las Vegas,Model_4,8,temperature_c | dew_point_c | station_pressure...,316.574484,437.417246,-0.132802,316.574484,437.417246,0.647228,11,1.817476
3,Las Vegas,Model_3,7,temperature_c | dew_point_c | station_pressure...,317.259232,437.693839,-0.134235,317.259232,437.693839,0.646782,11,1.785170
4,Phoenix,Model_4,8,temperature_c | dew_point_c | station_pressure...,766.095526,994.036377,0.068908,766.095526,994.036377,0.711584,11,1.435317
5,Phoenix,Model_3,7,temperature_c | dew_point_c | station_pressure...,782.408508,1007.280932,0.043931,782.408508,1007.280932,0.703847,11,1.365177
6,Phoenix,Model_1,5,temperature_c | dew_point_c | station_pressure...,794.273908,1017.405597,0.024615,794.273908,1017.405597,0.697863,11,1.381669
7,Phoenix,Model_2,6,temperature_c | dew_point_c | station_pressure...,818.738904,1051.983974,-0.042812,818.738904,1051.983974,0.676977,11,1.353753



BEST MODEL FOR EACH CITY (lowest held-out test final-energy RMSE):


,city,model,final_energy_rmse,final_energy_r2
0,Las Vegas,Model_2,423.045671,0.670028
4,Phoenix,Model_4,994.036377,0.711584


Overall best feature configuration: Model_2 (6 features), Las Vegas; test final-energy RMSE=423.046
Model_1: Las Vegas R² = 0.6659; Phoenix R² = 0.6979
Model_2: Las Vegas R² = 0.6700; Phoenix R² = 0.6770
Model_3: Las Vegas R² = 0.6468; Phoenix R² = 0.7038
Model_4: Las Vegas R² = 0.6472; Phoenix R² = 0.7116

Complete. Results: D:\Downloads\BTP\model_training\lstm_training_3\LSTM_GPU_TRAINING_RESULTS


## Interpretation

Select the best configuration by held-out **test final-energy** metrics, not by feature count, training score, or validation score alone. The saved comparison CSV makes it clear whether moisture, aerosol, or the nonlinear cooling response improved or worsened results.